# Stage 1 River — A02

This notebook owns exactly **48 work lanes** (approximate_producer). There is no science-authorization token or interlock. Dependency and byte-identity gates remain mandatory.


## Operator settings

The notebook always owns 48 logical lanes. The released worker default is Auto: use the minimum of usable logical CPUs, 48, and unfinished lanes. System RAM is observed for telemetry but never limits Auto or a valid positive worker override.


In [ ]:
REQUESTED_WORKERS = 0  # @param {type:"integer"}; 0 = Auto (min of usable logical CPUs, unfinished lanes, and 48); System RAM is observational only
TARGET_ITERATIONS = 3840000  # @param {type:"integer"}; approximate producers default to Tier 64 and may use a preregistered lower/higher tier
MINIMUM_QUALIFICATION_TIER = 16  # @param {type:"integer"}; fresh approximate runs use 16; a bounded repair may raise this to force continuation past an earlier passing checkpoint
EVALUATION_DEALS_PER_FAMILY_ROLE = 1024  # @param {type:"integer"}
try:
    REQUESTED_WORKERS = int(REQUESTED_WORKERS)
except (TypeError, ValueError) as error:
    raise ValueError("REQUESTED_WORKERS must be 0 for Auto or a positive integer.") from error
if REQUESTED_WORKERS < 0:
    raise ValueError("REQUESTED_WORKERS must be 0 for Auto or a positive integer.")
try:
    TARGET_ITERATIONS = int(TARGET_ITERATIONS)
    MINIMUM_QUALIFICATION_TIER = int(MINIMUM_QUALIFICATION_TIER)
    EVALUATION_DEALS_PER_FAMILY_ROLE = int(EVALUATION_DEALS_PER_FAMILY_ROLE)
except (TypeError, ValueError) as error:
    raise ValueError("Target iterations, minimum qualification tier, and evaluation deals must be integers.") from error
if True:
    if TARGET_ITERATIONS <= 0 or TARGET_ITERATIONS % 60000 != 0:
        raise ValueError("Approximate TARGET_ITERATIONS must be a positive whole preregistered tier.")
    TARGET_TIER = TARGET_ITERATIONS // 60000
else:
    TARGET_TIER = 0
if TARGET_TIER not in [16, 32, 48, 64, 80, 96, 112, 128]:
    raise ValueError("TARGET_ITERATIONS does not select one of this package's preregistered tiers.")
if True:
    if MINIMUM_QUALIFICATION_TIER not in [16, 32, 48, 64, 80, 96, 112, 128]:
        raise ValueError("MINIMUM_QUALIFICATION_TIER is not preregistered by this package.")
    if MINIMUM_QUALIFICATION_TIER > TARGET_TIER:
        raise ValueError("MINIMUM_QUALIFICATION_TIER cannot exceed the selected target tier.")
elif MINIMUM_QUALIFICATION_TIER != 0:
    raise ValueError("Non-approximate notebooks require MINIMUM_QUALIFICATION_TIER = 0.")
if not True and TARGET_ITERATIONS != 3840000:
    raise ValueError("This non-approximate notebook has an immutable target.")
if EVALUATION_DEALS_PER_FAMILY_ROLE not in [1024, 2048, 3072, 4096]:
    raise ValueError("Evaluation deals are not one of this package's preregistered sample sizes.")


## Phase 1 — Mount, bind, verify, and stage exact package bytes


In [ ]:
import hashlib
import json
import os
import shutil
import sys
import tempfile
import time
import zipfile
from pathlib import Path

from google.colab import drive

TOKEN = 'A02'
EXPECTED_LANE_COUNT = 48
DRIVE_RELATIVE_PATH = 'Poker Assistant Model/Stage 1 - River CFR and DCFR'
STATIC_MANIFEST_NAME = 'stage1_river_static_manifest_r22.json'
INPUT_PACKAGE_NAME = 'POKERNOW_ASSISTANT_STAGE1_RIVER_INPUTS_R22.zip'
INPUT_PACKAGE_SHA256 = '03b0e44af8082b24fe166a849e625d390da06c47b5d7bc1cb0b58c6e64b017ef'
LAUNCHER_GATE_NAME = 'stage1_river_launcher_gate_r22.json'
LAUNCHER_GATE_SHA256 = 'e54fe4b64838ce6cb629b21afa87d9d74bb70eeb5e28520e831cd9acc4dee4d1'
OPERATOR_NOTEBOOK_FILENAME = 'A02_Stage1_River_Approximate_RV0011-RV0018_8ROOTS_48LANES.ipynb'
DRIVE_ROOT_FOLDER_ID = '11x76L_iS6mgcSkow3S5MApGRuFvw02lE'
EXECUTION_MANIFEST_SELF_SHA256 = '290684f3f75e7f14f45f7698b24bf07efc6f102559dcb749e42a0a97dfd758e3'
ROOT_REGISTRY_SEMANTIC_SHA256 = '25e1106c4c00e9e3172abfec2426b1f0b54f930a9e21a3587693ce064006cc00'
STAGE_DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_RELATIVE_PATH
STATIC_MANIFEST_PATH = STAGE_DRIVE_ROOT / "manifests" / STATIC_MANIFEST_NAME
LAUNCHER_GATE_PATH = STAGE_DRIVE_ROOT / "manifests" / LAUNCHER_GATE_NAME
INPUT_PACKAGE_PATH = STAGE_DRIVE_ROOT / "inputs" / INPUT_PACKAGE_NAME

def canonical_json_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False).encode("utf-8")

def validate_self_hash(document):
    body = dict(document)
    claimed = body.pop("self_sha256", None)
    if claimed != hashlib.sha256(canonical_json_bytes(body)).hexdigest():
        raise RuntimeError("self-hashed release document mismatch")

def inspect_mounted_path():
    for _ in range(12):
        if STATIC_MANIFEST_PATH.is_file() and LAUNCHER_GATE_PATH.is_file() and INPUT_PACKAGE_PATH.is_file():
            return True
        time.sleep(5)
    return STATIC_MANIFEST_PATH.is_file() and LAUNCHER_GATE_PATH.is_file() and INPUT_PACKAGE_PATH.is_file()

mount_attempts = []
mounted_visible = STATIC_MANIFEST_PATH.is_file() and LAUNCHER_GATE_PATH.is_file() and INPUT_PACKAGE_PATH.is_file()
if not mounted_visible:
    try:
        drive.mount("/content/drive", force_remount=False)
        mount_attempts.append({"attempt": "initial", "exception": None})
    except Exception as error:
        mount_attempts.append({"attempt": "initial", "exception": f"{type(error).__name__}: {error}"})
    mounted_visible = inspect_mounted_path()
if not mounted_visible:
    try:
        drive.mount("/content/drive", force_remount=True)
        mount_attempts.append({"attempt": "forced_once", "exception": None})
    except Exception as error:
        mount_attempts.append({"attempt": "forced_once", "exception": f"{type(error).__name__}: {error}"})
    mounted_visible = inspect_mounted_path()
if not mounted_visible:
    raise RuntimeError(
        "The canonical mounted Stage path is unavailable after one initial mount and one "
        "conditional forced remount. This local release has no immutable Drive object ID, "
        "so an unbound API search is forbidden. Upload/publish the static package first."
    )

launcher_gate_raw = LAUNCHER_GATE_PATH.read_bytes()
if hashlib.sha256(launcher_gate_raw).hexdigest() != LAUNCHER_GATE_SHA256:
    raise RuntimeError("launcher gate byte identity mismatch")
launcher_gate = json.loads(launcher_gate_raw)
validate_self_hash(launcher_gate)
operator_rows = launcher_gate.get("operator_notebooks", [])
operator_row = next(
    (
        item for item in operator_rows
        if item.get("filename") == OPERATOR_NOTEBOOK_FILENAME
        and item.get("execution_token") == TOKEN
    ),
    None,
)
if (
    launcher_gate.get("revision") != 'STAGE21-BASELINE-TIER64-EVIDENCE-R22-R18-CANDIDATE'
    or launcher_gate.get("stage") != 'river'
    or launcher_gate.get("status") != "IMMUTABLE_LAUNCHER_GATE"
    or launcher_gate.get("drive_layout", {}).get("relative_path") != DRIVE_RELATIVE_PATH
    or launcher_gate.get("drive_layout", {}).get("root_folder_id") != DRIVE_ROOT_FOLDER_ID
    or launcher_gate.get("package", {}).get("filename") != INPUT_PACKAGE_NAME
    or launcher_gate.get("package", {}).get("sha256") != INPUT_PACKAGE_SHA256
    or launcher_gate.get("package_identity", {}).get("execution_manifest_self_sha256")
       != EXECUTION_MANIFEST_SELF_SHA256
    or launcher_gate.get("package_identity", {}).get("root_registry_semantic_sha256")
       != ROOT_REGISTRY_SEMANTIC_SHA256
    or operator_row is None
    or operator_row.get("lane_count") != 48
):
    raise RuntimeError("immutable launcher/package gate failed")

static_manifest_raw = STATIC_MANIFEST_PATH.read_bytes()
static_manifest = json.loads(static_manifest_raw)
validate_self_hash(static_manifest)
if (
    static_manifest.get("status") != "LOCAL_STATIC_PACKAGE_READY"
    or static_manifest.get("drive_layout", {}).get("relative_path") != DRIVE_RELATIVE_PATH
    or static_manifest.get("drive_layout", {}).get("root_folder_id") != DRIVE_ROOT_FOLDER_ID
    or static_manifest.get("package_identity", {}).get("execution_manifest_self_sha256")
       != EXECUTION_MANIFEST_SELF_SHA256
    or static_manifest.get("package_identity", {}).get("root_registry_semantic_sha256")
       != ROOT_REGISTRY_SEMANTIC_SHA256
    or static_manifest.get("input_package", {}).get("filename") != INPUT_PACKAGE_NAME
    or static_manifest.get("input_package", {}).get("sha256") != INPUT_PACKAGE_SHA256
    or static_manifest.get("launcher_gate", {}).get("filename") != LAUNCHER_GATE_NAME
    or static_manifest.get("launcher_gate", {}).get("sha256") != LAUNCHER_GATE_SHA256
):
    raise RuntimeError("static inventory manifest does not match launcher gate")

SCRATCH_PARENT = Path("/content/stage21_repaired_sources")
SCRATCH_PARENT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = Path(tempfile.mkdtemp(prefix=f"stage1_{TOKEN}_", dir=SCRATCH_PARENT))
LOCAL_INPUT_PACKAGE_PATH = SOURCE_ROOT / INPUT_PACKAGE_NAME
package_digest = hashlib.sha256()
package_size = 0
with INPUT_PACKAGE_PATH.open("rb") as package_handle, LOCAL_INPUT_PACKAGE_PATH.open("wb") as local_package_handle:
    for package_block in iter(lambda: package_handle.read(1024 * 1024), b""):
        package_digest.update(package_block)
        local_package_handle.write(package_block)
        package_size += len(package_block)
    local_package_handle.flush()
    os.fsync(local_package_handle.fileno())
if package_digest.hexdigest() != INPUT_PACKAGE_SHA256:
    raise RuntimeError("input package byte identity mismatch")
if package_size != int(launcher_gate["package"]["size"]):
    raise RuntimeError("input package byte size mismatch")
LOCAL_SCRATCH_PARENT = Path("/content/stage21_runtime_scratch") / 'STAGE21-BASELINE-TIER64-EVIDENCE-R22-R18-CANDIDATE' / f"stage1_{TOKEN}"
if str(LOCAL_SCRATCH_PARENT.resolve()).startswith("/content/drive/"):
    raise RuntimeError("runtime scratch resolved under mounted Drive")
with zipfile.ZipFile(LOCAL_INPUT_PACKAGE_PATH, "r") as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or any(
        name.startswith("/") or chr(92) in name or ".." in Path(name).parts for name in names
    ):
        raise RuntimeError("unsafe input package membership")
    archive.extractall(SOURCE_ROOT)
LOCAL_INPUT_PACKAGE_PATH.unlink()

execution = json.loads((SOURCE_ROOT / "contracts" / "execution_manifest.json").read_bytes())
if execution.get("self_sha256") != EXECUTION_MANIFEST_SELF_SHA256:
    raise RuntimeError("extracted execution-manifest identity mismatch")
dag_row = next((item for item in execution["notebooks"] if item["token"] == TOKEN), None)
if dag_row is None or dag_row.get("lane_count") != EXPECTED_LANE_COUNT:
    raise RuntimeError("notebook is not bound to an exact 48-lane DAG row")
if str(SOURCE_ROOT / "source") not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT / "source"))
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

RUNTIME_ROOT = STAGE_DRIVE_ROOT / "runtime_state" / 'STAGE21-BASELINE-TIER64-EVIDENCE-R22-R18-CANDIDATE'
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
probe = RUNTIME_ROOT / f".{TOKEN}.write_probe"
probe_raw = f"{TOKEN}:{time.time_ns()}".encode("utf-8")
probe.write_bytes(probe_raw)
if probe.read_bytes() != probe_raw:
    raise RuntimeError("mounted runtime-state write/readback failed")
probe.unlink()
print(json.dumps({"phase": 1, "token": TOKEN, "lane_count": 48, "mount_attempts": mount_attempts, "status": "VERIFIED"}, sort_keys=True), flush=True)


## Phase 2 — Validate roots, lane membership, dependency shape, and runtime inputs


In [ ]:
from stage21_baseline.roots import validate_registry
from stage21_baseline.runtime import LANE_PROGRESS_DETAIL_FIELDS, QUEUE_PROGRESS_LOG_FIELDS, resolve_worker_count

registry = json.loads((SOURCE_ROOT / "contracts" / "root_registry.json").read_bytes())
validate_registry(registry)
row = next(item for item in execution["notebooks"] if item["token"] == TOKEN)
if len(row["lanes"]) != 48 or len({item["lane_id"] for item in row["lanes"]}) != 48:
    raise RuntimeError("48-lane membership is incomplete or duplicated")
worker_resolution = resolve_worker_count(
    REQUESTED_WORKERS,
    unfinished_lane_count=len(row["lanes"]),
)
is_strict_aggregate = row["kind"] == "strict_aggregate"
print(json.dumps({
    "phase": 2,
    "token": TOKEN,
    "stage": 'river',
    "kind": row["kind"],
    "root_count": row["root_count"],
    "lane_count": row["lane_count"],
    "algorithms": execution["algorithms"],
    "seeds": execution["science_seeds"],
    "requested_workers": int(REQUESTED_WORKERS),
    "requested_workers_mode": worker_resolution["requested_workers_mode"],
    "detected_logical_cpu_count": worker_resolution["detected_logical_cpu_count"],
    "os_logical_cpu_count": worker_resolution["cpu_capacity_snapshot"]["os_logical_cpu_count"],
    "affinity_cpu_count": worker_resolution["affinity_cpu_count"],
    "cgroup_cpu_limit": worker_resolution["cgroup_cpu_limit"],
    "cgroup_cpu_snapshot": worker_resolution["cpu_capacity_snapshot"]["cgroup"],
    "runtime_worker_limit": worker_resolution["runtime_worker_limit"],
    "ram_worker_admission_enabled": False,
    "available_memory_bytes_at_resolution": worker_resolution["available_memory_bytes_at_resolution"],
    "memory_is_observational_only": True,
    "resolved_initial_live_worker_target": worker_resolution["resolved_live_worker_target"],
    "worker_count_clipped_by_cpu": worker_resolution["worker_count_clipped_by_cpu"],
    "worker_count_clipping_reasons": worker_resolution["clipping_reasons"],
    "worker_resolution_rule": worker_resolution["resolution_rule"],
    "depth_limit": execution["depth_limit"],
    "checkpoint_interval_iterations": (
        None
        if is_strict_aggregate
        else execution["restricted_exact_checkpoint_interval_iterations"]
        if row["root_kind"] == "restricted_exact"
        else execution["checkpoint_interval_iterations"]
    ),
    "checkpoint_generations_retained_per_lane": (
        None
        if is_strict_aggregate
        else execution["checkpoint_generations_retained_per_lane"]
    ),
    "checkpoint_semantics": (
        "NOT_APPLICABLE_TO_STRICT_AGGREGATE_SHARDS"
        if is_strict_aggregate
        else "DURABLE_PER_SOLVER_LANE"
    ),
    "progress_heartbeat_seconds": execution["progress_heartbeat_seconds"],
    "stall_warning_seconds": execution["stall_warning_seconds"],
    "progress_log_fields": list(QUEUE_PROGRESS_LOG_FIELDS),
    "per_lane_progress_detail_fields": list(LANE_PROGRESS_DETAIL_FIELDS),
    "persistent_aggregate_worker_pool": is_strict_aggregate,
    "one_task_per_process_lifetime": not is_strict_aggregate,
    "completed_lane_ram_release": (
        None
        if is_strict_aggregate
        else "durable readback -> compact handle -> gc/malloc_trim -> one-task worker reap"
    ),
    "aggregate_root_ram_release": (
        "close policy handles -> remove current-root materialization -> clear mutable continuation/runtime caches -> gc/malloc_trim; persistent process may accept another aggregate shard"
        if is_strict_aggregate
        else None
    ),
    "aggregate_post_shard_cleanup": (
        "release shard-local evidence and scratch before persistent worker reuse"
        if is_strict_aggregate
        else None
    ),
    "status": "PREFLIGHT_CONTRACTS_VALID",
}, sort_keys=True), flush=True)


## Phase 3 — Resume or execute the exact 48-lane queue

JSON heartbeats print real active/queued/completed/remaining counts, lane qualification counts, iteration/checkpoint ranges, percentage, ETA, RAM release, and stall warnings at least every 60 seconds.


In [ ]:
from stage21_baseline.runner import run_token

run_receipt = run_token(
    token=TOKEN,
    source_root=SOURCE_ROOT,
    stage_drive_root=STAGE_DRIVE_ROOT,
    runtime_root=RUNTIME_ROOT,
    local_scratch_root=LOCAL_SCRATCH_PARENT,
    requested_workers=REQUESTED_WORKERS,
    target_iterations=TARGET_ITERATIONS,
    target_tier=TARGET_TIER,
    minimum_qualification_tier=MINIMUM_QUALIFICATION_TIER,
    evaluation_deals_per_family_role=EVALUATION_DEALS_PER_FAMILY_ROLE,
)
print(json.dumps({
    "phase": 3,
    "token": TOKEN,
    "status": run_receipt["status"],
    "result": run_receipt["result"],
    "installed_result_validation_receipt": run_receipt["installed_result_validation_receipt"],
    "qualification_summary": run_receipt["queue"]["qualification_summary"],
    "worker_resolution": run_receipt["queue"]["worker_resolution"],
    "aggregate_staging": run_receipt.get("aggregate_staging"),
    "continuation_staging": run_receipt.get("continuation_staging"),
    "local_scratch_preflight": run_receipt.get("local_scratch_preflight"),
    "whole_token_writer_lease": run_receipt.get("whole_token_writer_lease"),
    "parent_memory_release": run_receipt["queue"]["parent_memory_release"],
}, sort_keys=True), flush=True)


## Phase 4 — Consume the installed-result receipt and perform a lightweight bound stat check


In [ ]:
from stage21_baseline.artifacts import lightweight_validate_installed_result

result_path = STAGE_DRIVE_ROOT / "returned_results" / row["result_filename"]
installed_receipt = run_receipt["installed_result_validation_receipt"]
result_stat = lightweight_validate_installed_result(
    result_path,
    installed_receipt,
    expected_stage=execution["stage"],
    expected_token=TOKEN,
)
if installed_receipt["lane_count"] != 48:
    raise RuntimeError("returned result does not contain 48 completed lane members")
print(json.dumps({
    "phase": 4,
    "token": TOKEN,
    "result_filename": result_path.name,
    "result_sha256": installed_receipt["sha256"],
    "result_size": result_stat["size"],
    "lane_count": installed_receipt["lane_count"],
    "installed_result_validation_receipt_self_sha256": installed_receipt["self_sha256"],
    "default_phase4_check": "LIGHTWEIGHT_BOUND_EXISTENCE_AND_STAT_ONLY",
    "status": run_receipt["status"] if row["kind"] == "strict_aggregate" else "INSTALLED_RESULT_RECEIPT_BOUND_UNADJUDICATED",
    "stage_adjudication_status": (
        run_receipt.get("stage_adjudication") or {}
    ).get("status"),
}, sort_keys=True), flush=True)
